# Лабораторная работа №3. Вариант 4.4: Online News Popularity

**Уровень 4 (продвинутый) — 9 баллов**

Специфика варианта:
- TF-IDF для url (извлечение домена)
- VIF для всех 60 признаков
- Optuna для XGBoost (100 trials)

In [1]:
import polars as pl
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# 1. Ленивая загрузка (Lazy API) и предобработка
lazy_df = pl.scan_csv('OnlineNewsPopularity.csv').with_columns([
    pl.col(' shares').log1p().alias('log_shares'),
    pl.col('url').str.extract(r'https?://(?:www\.)?([^/]+)').alias('domain')
]).filter(pl.col(' shares') > 0)

print(lazy_df.explain())
df = lazy_df.collect()

# TF-IDF для колонки url
tfidf = TfidfVectorizer(max_features=500, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['url'].to_list())
svd = TruncatedSVD(n_components=20, random_state=42)
tfidf_features = svd.fit_transform(tfidf_matrix)

for i in range(20):
    df = df.with_columns(pl.Series(f'tfidf_{i}', tfidf_features[:, i]))

 WITH_COLUMNS:
 [col(" shares").log1p().alias("log_shares"), col("url").str.extract(["https?://(?:www\.)?([^/]+)"]).alias("domain")] 
  Csv SCAN [OnlineNewsPopularity.csv]
  PROJECT */61 COLUMNS
  SELECTION: col(" shares") > dyn int: 0
  ESTIMATED ROWS: 43259


ComputeError: cannot compare string with numeric type (i32)

This error occurred in the following expression:
	[(col(" shares")) > (dyn int: 0)]


## 2. VIF и Отбор признаков (RFE)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

numerical_cols = [c for c in df.columns if c not in ['url', 'domain', ' shares', 'log_shares']]
X_num = df.select(numerical_cols).to_numpy()

vif_values = []
for i in range(X_num.shape[1]):
    X_other = np.delete(X_num, i, axis=1)
    y_other = X_num[:, i]
    lr = LinearRegression().fit(X_other, y_other)
    r2 = lr.score(X_other, y_other)
    vif = 1 / (1 - r2) if r2 < 1 else float('inf')
    vif_values.append(vif)

vif_df = pd.DataFrame({'Feature': numerical_cols, 'VIF': vif_values})
features_to_drop = vif_df[vif_df['VIF'] > 10]['Feature'].tolist()
print(f"Признаки на удаление (VIF > 10): {features_to_drop}")

df = df.drop(features_to_drop)
clean_num_cols = [c for c in numerical_cols if c not in features_to_drop]

# RFE
rf_rfe = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42, n_jobs=-1)
rfe = RFE(estimator=rf_rfe, n_features_to_select=20)
X_clean = df.select(clean_num_cols).to_numpy()
y = df['log_shares'].to_numpy()
rfe.fit(X_clean, y)
selected_features = [c for c, s in zip(clean_num_cols, rfe.support_) if s]

## 3. Кодирование категориальных признаков

In [ ]:
# 1. One-Hot Encoding
df_oh = df.to_dummies(columns=['domain'])

# 2. Target Encoding
alpha = 10
global_mean = df['log_shares'].mean()
cat_stats = df.group_by('domain').agg([
    pl.col('log_shares').mean().alias('cat_mean'),
    pl.len().alias('cat_count')
])
cat_stats = cat_stats.with_columns(
    ((pl.col('cat_mean') * pl.col('cat_count') + alpha * global_mean) / (pl.col('cat_count') + alpha)).alias('domain_target_enc')
)
df = df.join(cat_stats.select(['domain', 'domain_target_enc']), on='domain', how='left')

# 3. Frequency Encoding
freq = df.group_by('domain').agg((pl.len() / df.height).alias('domain_freq_enc'))
df = df.join(freq, on='domain', how='left')

## 4. Обучение моделей и Optuna

In [ ]:
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

X_final = df.select(selected_features + ['domain_target_enc']).to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X_final, y, test_size=0.2, random_state=42)

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0)
    }
    model = xgb.XGBRegressor(**params, random_state=42)
    model.fit(X_tr, y_tr)
    return np.sqrt(mean_squared_error(y_te, model.predict(X_te)))

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100)

print(f"Best params: {study.best_params}")
best_xgb = xgb.XGBRegressor(**study.best_params, random_state=42).fit(X_tr, y_tr)

## 5. SHAP и Статистические тесты

In [ ]:
import shap
from scipy.stats import shapiro
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

# SHAP
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_te[:1000])
shap.summary_plot(shap_values, X_te[:1000], feature_names=selected_features + ['domain_target_enc'])

# Статистические тесты (на остатках линейной модели)
lr = LinearRegression().fit(X_tr, y_tr)
residuals = y_te - lr.predict(X_te)

stat_shapiro, p_shapiro = shapiro(np.random.choice(residuals, 5000, replace=False))
print(f"Shapiro-Wilk p-value: {p_shapiro:.4f}")

bp_test = het_breuschpagan(residuals, sm.add_constant(X_te))
print(f"Breusch-Pagan p-value: {bp_test[1]:.4f}")

dw_stat = durbin_watson(residuals)
print(f"Durbin-Watson stat: {dw_stat:.4f}")

## 6. DuckDB SQL и Benchmark

In [ ]:
import duckdb
import time

con = duckdb.connect()
con.register('news', df.to_arrow())
query = """
    SELECT domain, AVG(log_shares) as avg_shares, COUNT(*) as cnt
    FROM news
    GROUP BY domain
    ORDER BY avg_shares DESC
    LIMIT 10
"""
print("DuckDB Query Result:")
print(con.execute(query).fetchdf())

# Benchmark
t0 = time.time()
_ = pl.scan_csv('OnlineNewsPopularity.csv').filter(pl.col(' shares') > 1000).collect()
t_pl = time.time() - t0

t0 = time.time()
df_pd = pd.read_csv('OnlineNewsPopularity.csv')
_ = df_pd[df_pd[' shares'] > 1000]
t_pd = time.time() - t0

print(f"\nPolars: {t_pl:.4f}s | Pandas: {t_pd:.4f}s | Ускорение: {t_pd/t_pl:.1f}x")